# Chapter 5 GRPO reward tracking

This notebook reproduces the Chapter 5 GRPO reward setup and logs the two reward groups that matter for the manuscript claim: binary correctness reward and auxiliary format reward. It saves CSV logs plus light and dark plot variants that can be dropped back into the book.

Default settings are chosen to be close to Chapter 5 while still being practical on a single Colab/GCP GPU. Increase `MAX_STEPS`, `TRAIN_SUBSET_SIZE`, or `MAX_COMPLETION_LENGTH` if you want a longer run.

In [ ]:
!pip install -U "trl" "transformers" "accelerate" "datasets" "peft" "bitsandbytes" "matplotlib" "pandas" "numpy>=2.0"

In [ ]:
import os
import re
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import GRPOConfig, GRPOTrainer

os.environ["WANDB_DISABLED"] = "true"

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = Path("/content/ch5_grpo_reward_tracking")
PLOT_DIR = OUTPUT_DIR / "plots"
RUN_NAME = "ch5-grpo-reward-tracking"

SEED = 42
TRAIN_SUBSET_SIZE = 1024
MAX_STEPS = 200
LOGGING_STEPS = 5
SAVE_STEPS = 100
NUM_GENERATIONS = 16
MAX_COMPLETION_LENGTH = 512
USE_4BIT = True
REPORT_TO = "none"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("This notebook expects a CUDA GPU runtime.")

COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print({
    "model": MODEL_NAME,
    "cuda_device": torch.cuda.get_device_name(0),
    "compute_dtype": str(COMPUTE_DTYPE),
    "max_steps": MAX_STEPS,
    "num_generations": NUM_GENERATIONS,
    "output_dir": str(OUTPUT_DIR),
})

In [ ]:
SYSTEM_PROMPT = """
Respond in the following format:

<reasoning>
...
</reasoning>
<answer>
...
</answer>
"""

XML_COT_FORMAT = """\
<reasoning>
{reasoning}
</reasoning>
<answer>
{answer}
</answer>
"""

def extract_xml_answer(text: str) -> str:
    answer = text.split("<answer>")[-1]
    answer = answer.split("</answer>")[0]
    return answer.strip()

def extract_hash_answer(text: str) -> str | None:
    if "####" not in text:
        return None
    return text.split("####", 1)[1].strip().replace(",", "").replace("$", "")

def get_completion_texts(completions) -> list[str]:
    texts = []
    for completion in completions:
        if isinstance(completion, list):
            texts.append(completion[0]["content"])
        elif isinstance(completion, dict):
            texts.append(completion["content"])
        else:
            texts.append(str(completion))
    return texts

def get_gsm8k_questions(split="train"):
    data = load_dataset("openai/gsm8k", "main", split=split)
    data = data.map(lambda x: {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": x["question"]},
        ],
        "answer": extract_hash_answer(x["answer"]),
    })
    data = data.filter(lambda x: x["answer"] is not None)
    return data

dataset = get_gsm8k_questions("train")
if TRAIN_SUBSET_SIZE is not None:
    n = min(TRAIN_SUBSET_SIZE, len(dataset))
    dataset = dataset.shuffle(seed=SEED).select(range(n))

print(dataset)
print(dataset[0]["prompt"][-1]["content"][:300], "...")
print("answer:", dataset[0]["answer"])

In [ ]:
def count_xml(text: str) -> float:
    count = 0.0
    if text.count("<reasoning>\n") == 1:
        count += 0.125
    if text.count("\n</reasoning>\n") == 1:
        count += 0.125
    if text.count("\n<answer>\n") == 1:
        count += 0.125
        count -= len(text.split("\n</answer>\n")[-1]) * 0.001
    if text.count("\n</answer>") == 1:
        count += 0.125
        count -= (len(text.split("\n</answer>")[-1]) - 1) * 0.001
    return count

def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    responses = get_completion_texts(completions)
    extracted_responses = [extract_xml_answer(r) for r in responses]
    return [2.0 if r == a else 0.0 for r, a in zip(extracted_responses, answer)]

def format_reward_func(completions, log_metric=None, **kwargs) -> list[float]:
    responses = get_completion_texts(completions)
    extracted_responses = [extract_xml_answer(r) for r in responses]

    strict_pattern = r"^<reasoning>\n.*?\n</reasoning>\n<answer>\n.*?\n</answer>\n$"
    soft_pattern = r"<reasoning>.*?</reasoning>\s*<answer>.*?</answer>"

    xml = [count_xml(r) for r in responses]
    soft = [0.5 if re.match(soft_pattern, r, flags=re.DOTALL) else 0.0 for r in responses]
    strict = [0.5 if re.match(strict_pattern, r, flags=re.DOTALL) else 0.0 for r in responses]
    integer = [0.5 if r.isdigit() else 0.0 for r in extracted_responses]
    total = [x + s + st + i for x, s, st, i in zip(xml, soft, strict, integer)]

    if log_metric is not None:
        log_metric("format/xmlcount_mean", float(np.mean(xml)))
        log_metric("format/soft_mean", float(np.mean(soft)))
        log_metric("format/strict_mean", float(np.mean(strict)))
        log_metric("format/int_mean", float(np.mean(integer)))

    return total

format_reward_func.__name__ = "format_reward_func"
correctness_reward_func.__name__ = "correctness_reward_func"

toy_completion = [[{"role": "assistant", "content": XML_COT_FORMAT.format(reasoning="2+2=4", answer="4")}]]
print("toy correctness", correctness_reward_func(None, toy_completion, ["4"]))
print("toy format", format_reward_func(toy_completion))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

quantization_config = None
if USE_4BIT:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    )

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=COMPUTE_DTYPE,
    quantization_config=quantization_config,
    device_map="auto",
    attn_implementation="sdpa",
)
model.config.use_cache = False

peft_config = LoraConfig(
    r=16,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "up_proj", "down_proj", "gate_proj"],
    task_type="CAUSAL_LM",
    lora_dropout=0.05,
)

training_args = GRPOConfig(
    output_dir=str(OUTPUT_DIR),
    run_name=RUN_NAME,
    learning_rate=5e-6,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_steps=max(1, int(0.1 * MAX_STEPS)),
    lr_scheduler_type="cosine",
    logging_steps=LOGGING_STEPS,
    logging_first_step=True,
    bf16=(COMPUTE_DTYPE is torch.bfloat16),
    fp16=(COMPUTE_DTYPE is torch.float16),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=NUM_GENERATIONS,
    num_generations=NUM_GENERATIONS,
    generation_batch_size=NUM_GENERATIONS,
    max_completion_length=MAX_COMPLETION_LENGTH,
    max_steps=MAX_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    max_grad_norm=0.1,
    report_to=REPORT_TO,
    log_on_each_node=False,
    remove_unused_columns=False,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[format_reward_func, correctness_reward_func],
    args=training_args,
    train_dataset=dataset,
    peft_config=peft_config,
)

print("Trainer ready")

In [ ]:
train_result = trainer.train()
print(train_result)

In [ ]:
log_history = pd.DataFrame(trainer.state.log_history)
log_path = OUTPUT_DIR / "training_log_history.csv"
log_history.to_csv(log_path, index=False)

print("saved", log_path)
print("columns:")
for col in log_history.columns:
    print(" -", col)

display(log_history.tail(10))

In [ ]:
def find_col(df: pd.DataFrame, candidates: list[str]) -> str | None:
    for candidate in candidates:
        if candidate in df.columns:
            return candidate
    lowered = {c.lower(): c for c in df.columns}
    for candidate in candidates:
        for lowered_col, original_col in lowered.items():
            if candidate.lower() in lowered_col:
                return original_col
    return None

correctness_col = find_col(log_history, [
    "rewards/correctness_reward_func/mean",
    "reward/correctness_reward_func",
    "correctness_reward_func",
])
format_col = find_col(log_history, [
    "rewards/format_reward_func/mean",
    "reward/format_reward_func",
    "format_reward_func",
])
total_reward_col = find_col(log_history, ["reward"])
reward_std_col = find_col(log_history, ["reward_std"])

print({
    "correctness_col": correctness_col,
    "format_col": format_col,
    "total_reward_col": total_reward_col,
    "reward_std_col": reward_std_col,
})

if correctness_col is None or format_col is None:
    raise RuntimeError("Could not find correctness/format reward metrics. Inspect printed columns above.")

metric_cols = ["step", correctness_col, format_col]
for optional_col in [total_reward_col, reward_std_col]:
    if optional_col and optional_col not in metric_cols:
        metric_cols.append(optional_col)

metrics = log_history[metric_cols].dropna(subset=[correctness_col, format_col], how="all").copy()
metrics = metrics.sort_values("step")
metrics_path = OUTPUT_DIR / "reward_component_metrics.csv"
metrics.to_csv(metrics_path, index=False)
display(metrics.tail())
print("saved", metrics_path)

In [ ]:
def smoothed(series: pd.Series, window: int = 5) -> pd.Series:
    return series.astype(float).rolling(window=window, min_periods=1).mean()

def set_theme(ax, theme: str):
    if theme == "dark":
        ax.set_facecolor("#0b0f14")
        ax.figure.set_facecolor("#0b0f14")
        ax.tick_params(colors="#e5e7eb")
        ax.xaxis.label.set_color("#e5e7eb")
        ax.yaxis.label.set_color("#e5e7eb")
        ax.title.set_color("#e5e7eb")
        for spine in ax.spines.values():
            spine.set_color("#9ca3af")
        ax.grid(color="#374151", alpha=0.35)
    else:
        ax.set_facecolor("#ffffff")
        ax.figure.set_facecolor("#ffffff")
        ax.grid(color="#d1d5db", alpha=0.55)

def save_fig(fig, stem: str, theme: str):
    png = PLOT_DIR / f"{stem}_{theme}.png"
    svg = PLOT_DIR / f"{stem}_{theme}.svg"
    fig.savefig(png, dpi=220, bbox_inches="tight", facecolor=fig.get_facecolor())
    fig.savefig(svg, bbox_inches="tight", facecolor=fig.get_facecolor())
    print("saved", png)
    print("saved", svg)

def plot_reward_components(theme="light"):
    fig, ax = plt.subplots(figsize=(8, 4.6))
    set_theme(ax, theme)
    steps = metrics["step"]
    c1 = "#111827" if theme == "light" else "#f9fafb"
    c2 = "#6b7280" if theme == "light" else "#9ca3af"
    ax.plot(steps, smoothed(metrics[correctness_col]), label="Correctness reward", color=c1, linewidth=2.4)
    ax.plot(steps, smoothed(metrics[format_col]), label="Format reward", color=c2, linewidth=2.4, linestyle="--")
    ax.set_xlabel("Training step")
    ax.set_ylabel("Mean reward component")
    ax.set_title("Chapter 5 GRPO reward components")
    ax.set_ylim(bottom=0)
    legend = ax.legend(frameon=False)
    if theme == "dark":
        for text in legend.get_texts():
            text.set_color("#e5e7eb")
    save_fig(fig, "ch5_grpo_reward_components", theme)
    return fig

def plot_reward_share(theme="light"):
    fig, ax = plt.subplots(figsize=(8, 4.6))
    set_theme(ax, theme)
    steps = metrics["step"]
    correctness = metrics[correctness_col].astype(float)
    fmt = metrics[format_col].astype(float)
    denom = correctness + fmt
    share = np.where(denom > 0, fmt / denom, np.nan)
    c = "#111827" if theme == "light" else "#f9fafb"
    ax.plot(steps, smoothed(pd.Series(share, index=metrics.index)), color=c, linewidth=2.4)
    ax.axhline(0.5, color="#6b7280" if theme == "light" else "#9ca3af", linestyle="--", linewidth=1.4)
    ax.set_xlabel("Training step")
    ax.set_ylabel("Format share of shaped reward")
    ax.set_title("How much of the observed reward comes from formatting?")
    ax.set_ylim(0, 1)
    save_fig(fig, "ch5_grpo_format_reward_share", theme)
    return fig

def plot_total_reward(theme="light"):
    fig, ax = plt.subplots(figsize=(8, 4.6))
    set_theme(ax, theme)
    steps = metrics["step"]
    if total_reward_col and total_reward_col in metrics:
        total = metrics[total_reward_col].astype(float)
    else:
        total = metrics[correctness_col].astype(float) + metrics[format_col].astype(float)
    c = "#111827" if theme == "light" else "#f9fafb"
    ax.plot(steps, smoothed(total), color=c, linewidth=2.4)
    ax.set_xlabel("Training step")
    ax.set_ylabel("Mean total reward")
    ax.set_title("Chapter 5 GRPO total shaped reward")
    ax.set_ylim(bottom=0)
    save_fig(fig, "ch5_grpo_total_reward", theme)
    return fig

for theme in ["light", "dark"]:
    plot_reward_components(theme)
    plot_reward_share(theme)
    plot_total_reward(theme)

In [ ]:
summary = {
    "final_step": int(metrics["step"].dropna().iloc[-1]),
    "final_correctness_reward_mean": float(metrics[correctness_col].dropna().iloc[-1]),
    "final_format_reward_mean": float(metrics[format_col].dropna().iloc[-1]),
    "max_steps": MAX_STEPS,
    "num_generations": NUM_GENERATIONS,
    "model": MODEL_NAME,
}
if total_reward_col and total_reward_col in metrics:
    summary["final_total_reward_metric"] = float(metrics[total_reward_col].dropna().iloc[-1])
if reward_std_col and reward_std_col in metrics:
    summary["final_reward_std"] = float(metrics[reward_std_col].dropna().iloc[-1])

summary_path = OUTPUT_DIR / "summary.json"
summary_path.write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print("saved", summary_path)

In [ ]:
# Optional: inspect a few post-training generations for qualitative examples.
def generate_answer(question: str, max_new_tokens: int = 256) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output_ids[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

examples = []
for row in dataset.select(range(min(3, len(dataset)))):
    completion = generate_answer(row["prompt"][-1]["content"])
    wrapped_completion = [[{"role": "assistant", "content": completion}]]
    examples.append({
        "question": row["prompt"][-1]["content"],
        "gold_answer": row["answer"],
        "completion": completion,
        "extracted_answer": extract_xml_answer(completion),
        "correctness_reward": correctness_reward_func(None, wrapped_completion, [row["answer"]])[0],
        "format_reward": format_reward_func(wrapped_completion)[0],
    })

examples_path = OUTPUT_DIR / "sample_generations.jsonl"
with examples_path.open("w") as f:
    for example in examples:
        f.write(json.dumps(example) + "\n")

display(pd.DataFrame(examples)[["gold_answer", "extracted_answer", "correctness_reward", "format_reward"]])
print("saved", examples_path)

In [ ]:
# Bundle the artifacts for download.
!cd /content && zip -r ch5_grpo_reward_tracking_artifacts.zip ch5_grpo_reward_tracking >/dev/null
print("artifact zip: /content/ch5_grpo_reward_tracking_artifacts.zip")